<a href="https://colab.research.google.com/github/rola2277/deep-research-agent-langgraph-project/blob/main/Rola_Fakeeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q \
    "langchain>=1.4" \
    "langgraph>=1.2.11" \
    "langchain-openai>=1.2" \
    beautifulsoup4 \
    requests \
    python-dotenv


# ============================================================
# 2. IMPORTS
# ============================================================

import asyncio
import functools
import getpass
import logging
import operator
import os
import re
import socket
import time
import uuid

from contextvars import ContextVar
from dataclasses import dataclass, field
from typing import Annotated, Any, List, Optional, TypedDict
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


# ============================================================
# 3. API KEY + MODEL SETUP
# ============================================================

api_key = ""

try:
    # Google Colab Secrets
    from google.colab import userdata

    try:
        api_key = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        api_key = ""

except ImportError:
    # Local environment
    from dotenv import load_dotenv

    load_dotenv()
    api_key = os.environ.get("OPENROUTER_API_KEY", "")


if not api_key:
    api_key = getpass.getpass(
        "Paste your OPENROUTER_API_KEY: "
    )


if not api_key:
    raise ValueError(
        "Please set OPENROUTER_API_KEY in Colab Secrets "
        "or enter it when prompted."
    )


MODEL_NAME = "deepseek/deepseek-v4-flash"

llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)


print("Model setup complete:", MODEL_NAME)


# ============================================================
# 4. OBSERVABILITY
# ============================================================

@dataclass
class Span:
    id: str
    name: str
    start_time: float
    level: int
    type: str = "span"
    parent: Optional["Span"] = None
    children: List["Span"] = field(default_factory=list)
    end_time: Optional[float] = None
    input: Any = None
    output: Any = None
    metadata: dict = field(default_factory=dict)
    usage: dict = field(default_factory=dict)
    model: Optional[str] = None


_current_span: ContextVar[Optional[Span]] = ContextVar(
    "current_span",
    default=None,
)


def print_tree(span: Span):
    duration = 0.0

    if span.end_time is not None:
        duration = (span.end_time - span.start_time) * 1000

    indent = "  " * span.level

    if span.level == 0:
        prefix = "=== TRACE: "
        suffix = " ==="
    else:
        prefix = "|-- "
        suffix = ""

    meta_parts = []

    if span.model:
        meta_parts.append(
            f"model={span.model}"
        )

    if span.usage.get("total_tokens"):
        meta_parts.append(
            f"tokens={span.usage['total_tokens']}"
        )

    if "cost_usd" in span.metadata:
        meta_parts.append(
            f"${span.metadata['cost_usd']:.4f}"
        )

    meta_str = (
        f" [{', '.join(meta_parts)}]"
        if meta_parts
        else ""
    )

    type_str = (
        f" [{span.type}]"
        if span.type != "span"
        else ""
    )

    print(
        f"{indent}{prefix}"
        f"{span.name}"
        f"{type_str}"
        f"{suffix}"
        f" ({duration:.2f}ms)"
        f"{meta_str}"
    )

    for child in span.children:
        print_tree(child)


def _make_span(
    span_name: str,
    span_type: str,
    args,
    kwargs,
) -> Span:

    parent = _current_span.get()

    level = (
        parent.level + 1
        if parent
        else 0
    )

    span = Span(
        id=str(uuid.uuid4())[:8],
        name=span_name,
        type=span_type,
        start_time=time.time(),
        level=level,
        parent=parent,
        input={
            "args": args,
            "kwargs": kwargs,
        },
    )

    if parent:
        parent.children.append(span)

    return span


def _finish_span(span: Span):

    span.end_time = time.time()

    if span.level == 0:
        print("\n" + "-" * 60)
        print_tree(span)
        print("-" * 60 + "\n")


def observe(name=None, as_type=None):

    def decorator(func):

        span_name = (
            name
            if isinstance(name, str)
            else func.__name__
        )

        span_type = as_type or "span"

        if asyncio.iscoroutinefunction(func):

            @functools.wraps(func)
            async def wrapper(*args, **kwargs):

                span = _make_span(
                    span_name,
                    span_type,
                    args,
                    kwargs,
                )

                token = _current_span.set(span)

                try:

                    result = await func(
                        *args,
                        **kwargs,
                    )

                    if span.output is None:
                        span.output = result

                    return result

                except Exception as e:

                    span.output = f"Error: {e}"

                    raise

                finally:

                    _finish_span(span)
                    _current_span.reset(token)

        else:

            @functools.wraps(func)
            def wrapper(*args, **kwargs):

                span = _make_span(
                    span_name,
                    span_type,
                    args,
                    kwargs,
                )

                token = _current_span.set(span)

                try:

                    result = func(
                        *args,
                        **kwargs,
                    )

                    if span.output is None:
                        span.output = result

                    return result

                except Exception as e:

                    span.output = f"Error: {e}"

                    raise

                finally:

                    _finish_span(span)
                    _current_span.reset(token)

        return wrapper

    if callable(name):
        func, name = name, None
        return decorator(func)

    return decorator


class LangfuseContext:

    def update_current_observation(self, **kwargs):

        span = _current_span.get()

        if not span:
            return

        if isinstance(
            kwargs.get("usage"),
            dict,
        ):
            span.usage.update(
                kwargs["usage"]
            )

        if "model" in kwargs:
            span.model = kwargs["model"]

        if isinstance(
            kwargs.get("metadata"),
            dict,
        ):
            span.metadata.update(
                kwargs["metadata"]
            )


langfuse_context = LangfuseContext()


# ============================================================
# 5. LOOP DETECTOR
# ============================================================

@dataclass
class LoopDetectionResult:
    is_looping: bool
    strategy: str
    message: str
    confidence: float


class LoopDetector:

    def __init__(
        self,
        exact_threshold: int = 2,
        fuzzy_threshold: float = 0.8,
        stagnation_window: int = 3,
    ):

        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window

        self.tool_history: list[
            tuple[str, str]
        ] = []

        self.output_history: list[str] = []

    def _jaccard_similarity(
        self,
        s1: str,
        s2: str,
    ) -> float:

        tokens1 = set(
            s1.lower().split()
        )

        tokens2 = set(
            s2.lower().split()
        )

        if not tokens1 and not tokens2:
            return 1.0

        if not tokens1 or not tokens2:
            return 0.0

        return (
            len(tokens1 & tokens2)
            /
            len(tokens1 | tokens2)
        )

    def check_tool_call(
        self,
        tool_name: str,
        tool_input: str,
    ) -> LoopDetectionResult:

        current = (
            tool_name,
            tool_input.strip(),
        )

        exact_count = sum(
            1
            for past_tool, past_input
            in self.tool_history
            if (
                past_tool,
                past_input.strip(),
            ) == current
        )

        if exact_count >= self.exact_threshold:

            self.tool_history.append(
                current
            )

            return LoopDetectionResult(
                is_looping=True,
                strategy="exact",
                confidence=1.0,
                message=(
                    f"Exact loop detected: "
                    f"'{tool_name}' repeated "
                    f"with identical arguments. "
                    f"Change the approach."
                ),
            )

        recent_history = (
            self.tool_history[-5:]
        )

        fuzzy_matches = sum(
            1
            for past_tool, past_input
            in recent_history
            if (
                past_tool == tool_name
                and
                self._jaccard_similarity(
                    tool_input,
                    past_input,
                )
                >= self.fuzzy_threshold
            )
        )

        if fuzzy_matches >= self.exact_threshold:

            self.tool_history.append(
                current
            )

            return LoopDetectionResult(
                is_looping=True,
                strategy="fuzzy",
                confidence=0.85,
                message=(
                    f"Fuzzy loop detected: "
                    f"'{tool_name}' received "
                    f"very similar inputs repeatedly."
                ),
            )

        self.tool_history.append(current)

        return LoopDetectionResult(
            is_looping=False,
            strategy="none",
            message="",
            confidence=0.0,
        )

    def check_output_stagnation(
        self,
        output: str,
    ) -> LoopDetectionResult:

        self.output_history.append(output)

        if len(self.output_history) < self.stagnation_window:

            return LoopDetectionResult(
                is_looping=False,
                strategy="none",
                message="",
                confidence=0.0,
            )

        recent = self.output_history[
            -self.stagnation_window:
        ]

        similarities = [
            self._jaccard_similarity(
                recent[i],
                recent[j],
            )
            for i in range(len(recent))
            for j in range(i + 1, len(recent))
        ]

        avg_similarity = (
            sum(similarities)
            / len(similarities)
            if similarities
            else 0
        )

        if avg_similarity >= self.fuzzy_threshold:

            return LoopDetectionResult(
                is_looping=True,
                strategy="stagnation",
                confidence=avg_similarity,
                message=(
                    f"Output stagnation detected: "
                    f"the last {self.stagnation_window} "
                    f"outputs are "
                    f"{avg_similarity:.0%} similar."
                ),
            )

        return LoopDetectionResult(
            is_looping=False,
            strategy="none",
            message="",
            confidence=0.0,
        )

    def reset(self):

        self.tool_history.clear()
        self.output_history.clear()


# ============================================================
# 6. WEB TOOLS
# ============================================================

logger = logging.getLogger(__name__)


def validate_url(url: str) -> bool:

    try:

        parsed = urlparse(url)

        if parsed.scheme not in (
            "http",
            "https",
        ):
            return False

        hostname = parsed.hostname

        if not hostname:
            return False

        try:

            ip_address = socket.gethostbyname(
                hostname
            )

        except socket.gaierror:

            return False

        parts = ip_address.split(".")

        if parts[0] == "10":
            return False

        if (
            parts[0] == "192"
            and parts[1] == "168"
        ):
            return False

        if (
            parts[0] == "172"
            and 16 <= int(parts[1]) <= 31
        ):
            return False

        if parts[0] == "127":
            return False

        if ip_address == "0.0.0.0":
            return False

        return True

    except Exception:

        return False


@tool("search_web")
def search_web(
    query: str,
    max_results: int = 5,
) -> str:
    """
    Search the web and return titles,
    links, and snippets.
    """

    url = "https://html.duckduckgo.com/html/"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:

        response = requests.post(
            url,
            data={"q": query},
            headers=headers,
            timeout=10,
        )

        response.raise_for_status()

    except Exception as e:

        logger.error(
            f"Search request failed: {e}"
        )

        return f"Search failed: {e}"

    soup = BeautifulSoup(
        response.text,
        "html.parser",
    )

    lines = []

    for result in soup.find_all(
        "div",
        class_="result",
        limit=max_results,
    ):

        title_tag = result.find(
            "a",
            class_="result__a",
        )

        snippet_tag = result.find(
            "a",
            class_="result__snippet",
        )

        if title_tag and snippet_tag:

            link = title_tag["href"]

            if validate_url(link):

                lines.append(
                    f"{len(lines) + 1}. "
                    f"{title_tag.get_text(strip=True)}\n"
                    f"   Link: {link}\n"
                    f"   Snippet: "
                    f"{snippet_tag.get_text(strip=True)}"
                )

    if not lines:

        return (
            f"No results found for "
            f"'{query}'."
        )

    return "\n\n".join(lines)


@tool("read_webpage")
def read_webpage(url: str) -> str:
    """
    Read a webpage and return cleaned text.
    """

    if not validate_url(url):

        return (
            "Error: Invalid or restricted URL."
        )

    try:

        if "example.com" in url:

            return (
                f"Simulated content for {url}."
            )

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10,
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser",
        )

        for script in soup(
            ["script", "style"]
        ):

            script.decompose()

        text = soup.get_text(
            separator="\n"
        )

        lines = (
            line.strip()
            for line in text.splitlines()
        )

        chunks = (
            phrase.strip()
            for line in lines
            for phrase in line.split("  ")
        )

        text = "\n".join(
            chunk
            for chunk in chunks
            if chunk
        )

        return text[:10000]

    except Exception as e:

        return (
            f"Error reading {url}: {e}"
        )


RESEARCH_TOOLS = [
    search_web,
    read_webpage,
]


# ============================================================
# 7. SHARED AGENT RUNNER
# ============================================================

@observe(
    name="agent_run",
    as_type="agent",
)
async def run_agent(
    agent,
    query: str,
    max_steps: int = 10,
) -> dict:

    result = await agent.ainvoke(
        {
            "messages": [
                ("user", query)
            ]
        },
        config={
            "recursion_limit":
                2 * max_steps + 1
        },
    )

    messages = result["messages"]

    answer = messages[-1].content

    total_tokens = sum(
        (
            m.usage_metadata or {}
        ).get(
            "total_tokens",
            0,
        )
        for m in messages
        if getattr(
            m,
            "usage_metadata",
            None,
        )
    )

    langfuse_context.update_current_observation(
        usage={
            "total_tokens":
                total_tokens
        },
        model=MODEL_NAME,
    )

    total_cost = sum(
        (
            m.response_metadata or {}
        ).get(
            "token_usage",
            {}
        ).get(
            "cost"
        )
        or 0.0
        for m in messages
    )

    langfuse_context.update_current_observation(
        metadata={
            "cost_usd":
                round(
                    total_cost,
                    6,
                )
        }
    )

    return {
        "answer": answer,
        "metadata": {
            "total_messages":
                len(messages),
            "total_tokens":
                total_tokens,
        },
    }


# ============================================================
# 8. PROMPTS
# ============================================================

RESEARCHER_PROMPT = """
You are the Researcher Agent in a multi-agent research workflow.

Your job is to gather reliable, relevant, and verifiable evidence.

Rules:
1. Use search_web to search for reliable sources.
2. Use read_webpage to inspect important source pages.
3. Prefer official documentation, academic papers,
   primary sources, and reputable institutions.
4. Never invent facts, citations, URLs, or statistics.
5. If a search fails, try a different search strategy.
6. Clearly separate verified facts from uncertainty.
7. Include source titles and URLs.
8. Keep research notes concise.

Return:
- Key findings
- Supporting evidence
- Source links
- Limitations
"""


ANALYST_PROMPT = """
You are the Analyst Agent in a multi-agent research workflow.

You receive research notes collected from independent research passes.

Produce a concise structured analysis:

1. Identify the main findings.
2. Compare evidence from the different research passes.
3. Identify agreements and contradictions.
4. Separate facts from interpretation.
5. Identify limitations and missing evidence.
6. Preserve useful source URLs.
7. Do not browse the web.
8. Do not invent facts or citations.

End with a section titled:

Evidence gaps
"""


WRITER_PROMPT = """
You are the Writer Agent in a multi-agent research workflow.

Turn the supplied research, analysis, and fact-check
results into a clear beginner-friendly final report.

Requirements:
1. Answer the original research question directly.
2. Use a descriptive title and clear headings.
3. Explain important terms.
4. Use only the supplied evidence.
5. Preserve useful source URLs.
6. Never invent facts or citations.
7. Mention uncertainty and evidence gaps.
8. Mention unresolved fact-check limitations.
9. End with a concise conclusion.
10. Include a Sources section.

Keep the report accurate, balanced, and readable.
"""


# ============================================================
# 9. BUILD AGENTS
# ============================================================

researcher = create_agent(
    model=llm,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            thread_limit=10,
            exit_behavior="end",
        )
    ],
)


analyst = create_agent(
    model=llm,
    tools=[],
    system_prompt=ANALYST_PROMPT,
)


writer = create_agent(
    model=llm,
    tools=[],
    system_prompt=WRITER_PROMPT,
)


planner = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are the Planning Agent. "
        "Decompose the user's research question into "
        "exactly two complementary research questions "
        "that can be investigated independently. "
        "Return exactly two lines, one question per line, "
        "with no numbering."
    ),
)


fact_checker = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a strict Fact-Checker Agent. "
        "Review the supplied research notes, analysis, "
        "and source links. Look for unsupported claims, "
        "contradictions, missing evidence, incorrect "
        "reasoning, and overclaims. "
        "At the end return exactly one final line: "
        "VERDICT: PASS or VERDICT: FAIL."
    ),
)


print(
    "Agents created:",
    "researcher, analyst, writer, planner, fact_checker"
)


# ============================================================
# 10. MULTI-AGENT PIPELINE
# ============================================================

async def run_pipeline(
    query: str,
) -> dict:

    MAX_RETRIES = 2
    MAX_STEPS = 20

    detector = LoopDetector()

    # --------------------------------------------------------
    # Shared state
    # --------------------------------------------------------

    class PipelineState(TypedDict):

        query: str
        report: str

        sub_queries: list[str]

        research_results: Annotated[
            list[str],
            operator.add,
        ]

        research_notes: str
        analysis: str

        fact_check: str
        fact_check_passed: bool

        retry_count: int

        step_count: Annotated[
            int,
            operator.add,
        ]

        loop_warnings: Annotated[
            list[str],
            operator.add,
        ]


    # ========================================================
    # Planner
    # ========================================================

    async def planner_node(
        state: PipelineState,
    ) -> dict:

        result = await run_agent(
            planner,
            (
                "Original research question:\n"
                f"{state['query']}\n\n"
                "Split it into exactly two complementary "
                "research questions."
            ),
            5,
        )

        lines = [
            line.strip()
            for line in result["answer"].splitlines()
            if line.strip()
        ]

        lines = [
            re.sub(
                r"^\d+[\).\s-]+",
                "",
                line,
            ).strip()
            for line in lines
        ]

        if len(lines) < 2:

            lines = [
                state["query"],
                (
                    f"{state['query']} - "
                    "evidence and limitations"
                ),
            ]

        return {
            "sub_queries":
                lines[:2],
            "step_count": 1,
            "loop_warnings": [],
        }


    # ========================================================
    # Research A
    # ========================================================

    async def researcher_a_node(
        state: PipelineState,
    ) -> dict:

        sub_query = (
            state["sub_queries"][0]
        )

        check = detector.check_tool_call(
            "research_branch_a",
            sub_query,
        )

        warnings = []

        if check.is_looping:
            warnings.append(
                check.message
            )

        result = await run_agent(
            researcher,
            (
                "Research question:\n"
                f"{sub_query}\n\n"
                "Use search_web and read_webpage. "
                "Find multiple reliable sources, "
                "inspect important pages, and provide URLs."
            ),
            10,
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        if stagnation.is_looping:
            warnings.append(
                stagnation.message
            )

        return {
            "research_results": [
                "RESEARCH PASS A\n"
                + result["answer"]
            ],
            "step_count": 1,
            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Research B
    # ========================================================

    async def researcher_b_node(
        state: PipelineState,
    ) -> dict:

        sub_query = (
            state["sub_queries"][1]
        )

        check = detector.check_tool_call(
            "research_branch_b",
            sub_query,
        )

        warnings = []

        if check.is_looping:
            warnings.append(
                check.message
            )

        result = await run_agent(
            researcher,
            (
                "Research question:\n"
                f"{sub_query}\n\n"
                "Use a different search strategy from Research Pass A. "
                "Find reliable and preferably independent sources. "
                "Verify important claims and provide URLs."
            ),
            10,
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        if stagnation.is_looping:
            warnings.append(
                stagnation.message
            )

        return {
            "research_results": [
                "RESEARCH PASS B\n"
                + result["answer"]
            ],
            "step_count": 1,
            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Combine Research
    # ========================================================

    async def combine_research_node(
        state: PipelineState,
    ) -> dict:

        combined_notes = (
            "\n\n====================\n\n"
            .join(
                state["research_results"]
            )
        )

        return {
            "research_notes":
                combined_notes,
            "step_count": 1,
            "loop_warnings": [],
        }


    # ========================================================
    # Analyst
    # ========================================================

    async def analyst_node(
        state: PipelineState,
    ) -> dict:

        result = await run_agent(
            analyst,
            (
                "Original question:\n"
                f"{state['query']}\n\n"
                "Research notes:\n"
                f"{state['research_notes']}\n\n"
                "Compare the research results and "
                "produce a structured evidence-based analysis."
            ),
            10,
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        warnings = []

        if stagnation.is_looping:
            warnings.append(
                stagnation.message
            )

        return {
            "analysis":
                result["answer"],
            "step_count": 1,
            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Fact Checker
    # ========================================================

    async def fact_checker_node(
        state: PipelineState,
    ) -> dict:

        result = await run_agent(
            fact_checker,
            (
                "Original question:\n"
                f"{state['query']}\n\n"
                "Research:\n"
                f"{state['research_notes']}\n\n"
                "Analysis:\n"
                f"{state['analysis']}\n\n"
                "Check important claims, contradictions, "
                "missing evidence, and unsupported conclusions.\n"
                "End with exactly:\n"
                "VERDICT: PASS\n"
                "or\n"
                "VERDICT: FAIL"
            ),
            8,
        )

        passed = (
            "VERDICT: PASS"
            in result["answer"].upper()
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        warnings = []

        if stagnation.is_looping:

            warnings.append(
                stagnation.message
            )

            passed = False

        return {
            "fact_check":
                result["answer"],

            "fact_check_passed":
                passed,

            "step_count": 1,

            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Retry Research
    # ========================================================

    async def retry_research_node(
        state: PipelineState,
    ) -> dict:

        retry_number = (
            state["retry_count"] + 1
        )

        check = detector.check_tool_call(
            "research_retry",
            (
                f"{state['query']} "
                f"retry {retry_number}"
            ),
        )

        warnings = []

        if check.is_looping:
            warnings.append(
                check.message
            )

        result = await run_agent(
            researcher,
            (
                "Original question:\n"
                f"{state['query']}\n\n"

                "Previous research:\n"
                f"{state['research_notes']}\n\n"

                "Previous analysis:\n"
                f"{state['analysis']}\n\n"

                "Fact-check result:\n"
                f"{state['fact_check']}\n\n"

                "The fact checker was not satisfied.\n"
                "Perform a new research pass.\n"
                "Do NOT repeat the previous search strategy.\n"
                "Find stronger or independent evidence.\n"
                "Verify important claims and provide source URLs."
            ),
            10,
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        if stagnation.is_looping:
            warnings.append(
                stagnation.message
            )

        updated_notes = (
            state["research_notes"]
            + "\n\n"
            + "================ RETRY ================\n\n"
            + result["answer"]
        )

        return {
            "research_notes":
                updated_notes,

            "retry_count":
                retry_number,

            "step_count":
                1,

            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Re-analysis
    # ========================================================

    async def reanalysis_node(
        state: PipelineState,
    ) -> dict:

        result = await run_agent(
            analyst,
            (
                "Original question:\n"
                f"{state['query']}\n\n"

                "Updated research:\n"
                f"{state['research_notes']}\n\n"

                "Previous fact-check:\n"
                f"{state['fact_check']}\n\n"

                "Re-analyze the updated evidence. "
                "Correct unsupported claims and strengthen the analysis."
            ),
            10,
        )

        stagnation = detector.check_output_stagnation(
            result["answer"]
        )

        warnings = []

        if stagnation.is_looping:
            warnings.append(
                stagnation.message
            )

        return {
            "analysis":
                result["answer"],

            "step_count":
                1,

            "loop_warnings":
                warnings,
        }


    # ========================================================
    # Writer
    # ========================================================

    async def writer_node(
        state: PipelineState,
    ) -> dict:

        result = await run_agent(
            writer,
            (
                "Original question:\n"
                f"{state['query']}\n\n"

                "Research notes:\n"
                f"{state['research_notes']}\n\n"

                "Analysis:\n"
                f"{state['analysis']}\n\n"

                "Fact-check:\n"
                f"{state['fact_check']}\n\n"

                "Write the final beginner-friendly report."
            ),
            10,
        )

        return {
            "report":
                result["answer"],

            "step_count":
                1,

            "loop_warnings":
                [],
        }


    # ========================================================
    # CONDITIONAL ROUTER
    # ========================================================

    def route_after_fact_check(
        state: PipelineState,
    ) -> str:

        # PASS -> Writer
        if state["fact_check_passed"]:
            return "writer"

        # Maximum retry count
        if (
            state["retry_count"]
            >= MAX_RETRIES
        ):
            return "writer"

        # Maximum pipeline steps
        if (
            state["step_count"]
            >= MAX_STEPS
        ):
            return "writer"

        # FAIL -> retry research
        return "retry_research"


    # ========================================================
    # BUILD GRAPH
    # ========================================================

    graph = StateGraph(
        PipelineState
    )

    graph.add_node(
        "planner",
        planner_node,
    )

    graph.add_node(
        "researcher_a",
        researcher_a_node,
    )

    graph.add_node(
        "researcher_b",
        researcher_b_node,
    )

    graph.add_node(
        "combine_research",
        combine_research_node,
    )

    graph.add_node(
        "analyst",
        analyst_node,
    )

    graph.add_node(
        "fact_checker",
        fact_checker_node,
    )

    graph.add_node(
        "retry_research",
        retry_research_node,
    )

    graph.add_node(
        "reanalysis",
        reanalysis_node,
    )

    graph.add_node(
        "writer",
        writer_node,
    )


    # START -> Planner
    graph.add_edge(
        START,
        "planner",
    )


    # Planner -> TWO PARALLEL RESEARCHERS
    graph.add_edge(
        "planner",
        "researcher_a",
    )

    graph.add_edge(
        "planner",
        "researcher_b",
    )


    # Parallel Research -> Combine
    graph.add_edge(
        "researcher_a",
        "combine_research",
    )

    graph.add_edge(
        "researcher_b",
        "combine_research",
    )


    # Combine -> Analyst -> Fact Checker
    graph.add_edge(
        "combine_research",
        "analyst",
    )

    graph.add_edge(
        "analyst",
        "fact_checker",
    )


    # Conditional route
    graph.add_conditional_edges(
        "fact_checker",
        route_after_fact_check,
        {
            "writer":
                "writer",

            "retry_research":
                "retry_research",
        },
    )


    # Retry loop
    graph.add_edge(
        "retry_research",
        "reanalysis",
    )

    graph.add_edge(
        "reanalysis",
        "fact_checker",
    )


    # Writer -> END
    graph.add_edge(
        "writer",
        END,
    )


    # Compile
    pipeline = graph.compile()


    # ========================================================
    # INITIAL STATE
    # ========================================================

    initial_state: PipelineState = {

        "query":
            query,

        "report":
            "",

        "sub_queries":
            [],

        "research_results":
            [],

        "research_notes":
            "",

        "analysis":
            "",

        "fact_check":
            "",

        "fact_check_passed":
            False,

        "retry_count":
            0,

        "step_count":
            0,

        "loop_warnings":
            [],
    }


    # ========================================================
    # RUN
    # ========================================================

    result = await pipeline.ainvoke(
        initial_state
    )


    # ========================================================
    # VISIBLE RELIABILITY WARNINGS
    # ========================================================

    if result.get("loop_warnings"):

        print("\n")
        print("=" * 70)
        print("LOOP / STAGNATION WARNINGS")
        print("=" * 70)

        for warning in result["loop_warnings"]:

            print(
                "-",
                warning,
            )


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "answer":
            result["report"],

        "metadata": {

            "graph_nodes":
                9,

            "architecture":
                (
                    "planner + parallel research "
                    "+ analyst + fact-check retry loop"
                ),

            "executed_retries":
                result["retry_count"],

            "steps":
                result["step_count"],

            "fact_check_passed":
                result["fact_check_passed"],

            "loop_warnings":
                len(
                    result["loop_warnings"]
                ),
        },
    }


# ============================================================
# 11. RUN THE PROJECT
# ============================================================

query = "Compare RAG and fine-tuning"

print("\n")
print("=" * 70)
print("STARTING MULTI-AGENT RESEARCH PIPELINE")
print("=" * 70)
print("Question:", query)
print("=" * 70)


pipeline_result = await run_pipeline(
    query
)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL REPORT")
print("=" * 70)

print(
    pipeline_result["answer"]
)


# ============================================================
# 13. METADATA
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE METADATA")
print("=" * 70)

for key, value in pipeline_result[
    "metadata"
].items():

    print(
        f"{key}: {value}"
    )


print("\n")
print("=" * 70)
print("PIPELINE COMPLETED")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 13.5 MB/s eta 0:00:00
